%% [markdown]
# # Image Baseline — Fraud Detection

# ResNet-18 fine-tuned for binary document/image fraud classification.

# Pipeline

1. Load & validate CSV splits
2. Inspect class balance and group overlap
 3. Two-stage training:
   - Stage 1: frozen backbone, warm up the head (5 epochs)
   - Stage 2: full fine-tuning with cosine LR decay
4. Tune decision threshold on validation set
5. Test evaluation
6. Multi-seed robustness check
7. Subgroup analysis (catch dataset-level shortcuts)
8. GradCAM sanity check (verify the model attends to fraud regions)

In [7]:
import sys
sys.path.insert(0, r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\src")

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

from image_baseline_test import (
    load_image_splits, print_split_summary, build_dataloaders,
    build_model, unfreeze_backbone, make_loss_fn,
    fit_model, run_test_evaluation,
    predict_probs, compute_metrics, tune_threshold, evaluate,
    subgroup_metrics,
    GradCAM,
)

In [8]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
 
 
SEED = 42
set_seed(SEED)
 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cpu


In [12]:
# Base project root (one level up from /notebook)
PROJECT_ROOT = Path().resolve().parent

DATA_DIR    = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "notebook" / "results" / "image_baseline"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = DATA_DIR / "image_train_group_test.csv"
VAL_CSV   = DATA_DIR / "image_val_group_test.csv"
TEST_CSV  = DATA_DIR / "image_test_group_test.csv"

In [13]:
BATCH_SIZE   = 32
NUM_WORKERS  = 0

In [14]:
splits = load_image_splits(
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    test_csv=TEST_CSV,
)
 
print_split_summary(splits.train_df, splits.val_df, splits.test_df)


TRAIN
Rows: 7182
Groups: 953
image_class
forged       3731
bona_fide    3451
Name: count, dtype: int64
image_class
forged       51.95
bona_fide    48.05
Name: proportion, dtype: float64

VAL
Rows: 1508
Groups: 204
image_class
forged       765
bona_fide    743
Name: count, dtype: int64
image_class
forged       50.73
bona_fide    49.27
Name: proportion, dtype: float64

TEST
Rows: 1474
Groups: 205
image_class
bona_fide    739
forged       735
Name: count, dtype: int64
image_class
bona_fide    50.14
forged       49.86
Name: proportion, dtype: float64

GROUP OVERLAP
  train_val_overlap: 0
  train_test_overlap: 0
  val_test_overlap: 0


In [15]:
train_loader, val_loader, test_loader = build_dataloaders(
    splits=splits,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
 
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

Train batches : 225
Val batches   : 48
Test batches  : 47


In [16]:
STAGE1_EPOCHS   = 5
STAGE1_LR       = 1e-3   # higher LR is fine since only the head is trained
STAGE1_PATIENCE = 5
 
model = build_model(pretrained=True, freeze_backbone=True, dropout=0.3).to(device)
loss_fn = make_loss_fn(splits.train_df, device=device)
 
optimizer_s1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=STAGE1_LR,
    weight_decay=1e-4,
)
 
# No scheduler needed for such a short warmup
model, history_s1 = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer_s1,
    device=device,
    epochs=STAGE1_EPOCHS,
    threshold=0.5,
    early_stopping_patience=STAGE1_PATIENCE,
    monitor_metric="val_roc_auc",
    model_save_path=RESULTS_DIR / "stage1_best.pt",
)
 
print("\nStage 1 history:")
print(history_s1.to_string(index=False))

  pos_weight = 0.925  (n_neg=3451, n_pos=3731)
Epoch 01 | lr=1.00e-03 | train_loss=0.4251 | val_loss=0.3420 | val_f1=0.8343 | val_roc_auc=0.9208
  ✓ Saved best model (epoch 1, val_roc_auc=0.9208)
Epoch 02 | lr=1.00e-03 | train_loss=0.3502 | val_loss=0.3189 | val_f1=0.8542 | val_roc_auc=0.9344
  ✓ Saved best model (epoch 2, val_roc_auc=0.9344)
Epoch 03 | lr=1.00e-03 | train_loss=0.3333 | val_loss=0.3050 | val_f1=0.8413 | val_roc_auc=0.9420
  ✓ Saved best model (epoch 3, val_roc_auc=0.9420)
Epoch 04 | lr=1.00e-03 | train_loss=0.3278 | val_loss=0.2891 | val_f1=0.8638 | val_roc_auc=0.9431
  ✓ Saved best model (epoch 4, val_roc_auc=0.9431)
Epoch 05 | lr=1.00e-03 | train_loss=0.3241 | val_loss=0.2860 | val_f1=0.8670 | val_roc_auc=0.9449
  ✓ Saved best model (epoch 5, val_roc_auc=0.9449)
Restored best weights (val_roc_auc=0.9449).

Stage 1 history:
 epoch    lr  train_loss  val_loss  val_accuracy  val_precision  val_recall   val_f1  val_roc_auc
     1 0.001    0.425116  0.342042      0.825597

## Stage 2 — Full fine-tuning (backbone unfrozen)

Use a **differential learning rate**: backbone gets a much smaller LR than the head because its weights are already good. CosineAnnealingLR will smoothly decay the LR to near-zero over the training budget.


In [18]:
STAGE2_EPOCHS   = 20
BACKBONE_LR     = 1e-5   # ← 10–100x lower than head LR
HEAD_LR         = 1e-4
STAGE2_PATIENCE = 6      # generous patience because val metrics can be noisy
 
unfreeze_backbone(model)
 
# Differential LR: separate param groups for backbone vs head
backbone_params = [p for n, p in model.named_parameters() if "fc" not in n]
head_params     = list(model.fc.parameters())
 
optimizer_s2 = torch.optim.Adam(
    [
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params,     "lr": HEAD_LR},
    ],
    weight_decay=1e-4,
)
 
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_s2,
    T_max=STAGE2_EPOCHS,
    eta_min=1e-7,
)
 
model, history_s2 = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer_s2,
    device=device,
    epochs=STAGE2_EPOCHS,
    threshold=0.5,
    early_stopping_patience=STAGE2_PATIENCE,
    monitor_metric="val_roc_auc",
    scheduler=scheduler,
    model_save_path=RESULTS_DIR / "stage2_best.pt",
)

Backbone unfrozen — all parameters are now trainable.
Epoch 01 | lr=1.00e-05 | train_loss=0.2737 | val_loss=0.2212 | val_f1=0.8923 | val_roc_auc=0.9627
  ✓ Saved best model (epoch 1, val_roc_auc=0.9627)
Epoch 02 | lr=9.94e-06 | train_loss=0.2283 | val_loss=0.2123 | val_f1=0.9074 | val_roc_auc=0.9645
  ✓ Saved best model (epoch 2, val_roc_auc=0.9645)
Epoch 03 | lr=9.76e-06 | train_loss=0.2177 | val_loss=0.1973 | val_f1=0.9099 | val_roc_auc=0.9656
  ✓ Saved best model (epoch 3, val_roc_auc=0.9656)
Epoch 04 | lr=9.46e-06 | train_loss=0.2046 | val_loss=0.1960 | val_f1=0.9013 | val_roc_auc=0.9659
  ✓ Saved best model (epoch 4, val_roc_auc=0.9659)
Epoch 05 | lr=9.05e-06 | train_loss=0.1953 | val_loss=0.1908 | val_f1=0.9132 | val_roc_auc=0.9670
  ✓ Saved best model (epoch 5, val_roc_auc=0.9670)
Epoch 06 | lr=8.55e-06 | train_loss=0.1912 | val_loss=0.1911 | val_f1=0.9058 | val_roc_auc=0.9675
  ✓ Saved best model (epoch 6, val_roc_auc=0.9675)
Epoch 07 | lr=7.96e-06 | train_loss=0.1878 | val_los

In [19]:
history = pd.concat([history_s1, history_s2], ignore_index=True)
history["epoch_global"] = range(1, len(history) + 1)
history.to_csv(RESULTS_DIR / "training_history.csv", index=False)
 
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
 
axes[0].plot(history["epoch_global"], history["train_loss"], label="train")
axes[0].plot(history["epoch_global"], history["val_loss"],   label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
 
axes[1].plot(history["epoch_global"], history["val_f1"],      label="F1")
axes[1].plot(history["epoch_global"], history["val_roc_auc"], label="ROC-AUC")
axes[1].set_title("Val metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()
 
axes[2].plot(history["epoch_global"], history["lr"])
axes[2].set_title("Learning rate")
axes[2].set_xlabel("Epoch")
axes[2].set_yscale("log")
 
# Vertical line separating Stage 1 from Stage 2
n_s1 = len(history_s1)
for ax in axes:
    ax.axvline(x=n_s1 + 0.5, color="gray", linestyle="--", alpha=0.6, label="unfreeze")
 
fig.tight_layout()
fig.savefig(RESULTS_DIR / "training_curves.png", dpi=150)
plt.close(fig)
print("Saved training_curves.png")

Saved training_curves.png


In [20]:
history = pd.concat([history_s1, history_s2], ignore_index=True)
history["epoch_global"] = range(1, len(history) + 1)
history.to_csv(RESULTS_DIR / "training_history.csv", index=False)

n_s1 = len(history_s1)
ep   = history["epoch_global"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# — Loss —
axes[0].plot(ep, history["train_loss"], label="train", color="#4C72B0")
axes[0].plot(ep, history["val_loss"],   label="val",   color="#DD8452")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

# — Val metrics —
axes[1].plot(ep, history["val_f1"],      label="F1",      color="#55A868")
axes[1].plot(ep, history["val_roc_auc"], label="ROC-AUC", color="#C44E52")
axes[1].set_ylim(0.75, 1.01)
axes[1].set_title("Validation metrics"); axes[1].set_xlabel("Epoch"); axes[1].legend()

# — Learning rate —
axes[2].plot(ep, history["lr"], color="#8172B2")
axes[2].set_title("Learning rate"); axes[2].set_xlabel("Epoch")
axes[2].set_yscale("log")

for ax in axes:
    ax.axvline(x=n_s1 + 0.5, color="gray", linestyle="--", alpha=0.7)
    ax.spines[["top", "right"]].set_visible(False)

axes[0].text(n_s1 / 2, axes[0].get_ylim()[1] * 0.98, "Stage 1",
             ha="center", va="top", fontsize=9, color="gray")
axes[0].text(n_s1 + (len(ep) - n_s1) / 2, axes[0].get_ylim()[1] * 0.98, "Stage 2",
             ha="center", va="top", fontsize=9, color="gray")

fig.suptitle("Training overview", fontsize=13)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_10196\3549917154.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
val_y_true, val_y_prob = predict_probs(model, val_loader, device)
best_threshold, sweep_df = tune_threshold(val_y_true, val_y_prob, metric="f1")
sweep_df.to_csv(RESULTS_DIR / "threshold_sweep.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Precision / Recall / F1 vs threshold
axes[0].plot(sweep_df["threshold"], sweep_df["precision"], label="Precision", color="#4C72B0")
axes[0].plot(sweep_df["threshold"], sweep_df["recall"],    label="Recall",    color="#DD8452")
axes[0].plot(sweep_df["threshold"], sweep_df["f1"],        label="F1",        color="#55A868", linewidth=2)
axes[0].axvline(x=best_threshold, color="red", linestyle="--", label=f"Best ({best_threshold:.2f})")
axes[0].set_xlabel("Threshold"); axes[0].set_ylabel("Score")
axes[0].set_title("Precision / Recall / F1 vs threshold")
axes[0].legend(); axes[0].spines[["top", "right"]].set_visible(False)

# ROC curve
fpr, tpr, _ = roc_curve(val_y_true, val_y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color="#C44E52", lw=2, label=f"Val ROC-AUC = {roc_auc:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve (validation)")
axes[1].legend(); axes[1].spines[["top", "right"]].set_visible(False)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "threshold_and_roc.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nSelected threshold: {best_threshold:.2f}")